In [126]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from scipy.stats import ks_2samp, anderson_ksamp

import warnings
from tqdm import tqdm
tqdm.pandas()


In [ ]:
import numpy as np
import pandas as pd
from galpy.potential import MWPotential2014
from galpy.orbit import Orbit
from astropy import units as u

def compute_orbit_params(RA, Dec, plx, pmRA, pmDec, Vlos, ro=8.125, vo=220.):
    try:
        distance = 1.0 / plx  # distance in kpc

        o = Orbit([RA, Dec, distance, pmRA, pmDec, Vlos],
                  radec=True, ro=ro, vo=vo)

        ts = np.linspace(0, 10, 1000) * u.Gyr
        o.integrate(ts, MWPotential2014)

        return pd.Series({
            'rapo': o.rap(),
            'rperi': o.rperi(),
            'eccentricity': o.e(),
            'zmax': o.zmax(),
            'E': o.E(),
            'ER': o.ER(),
            'Ez': o.Ez(),
            'LcE': o.LcE(),
            'Lz': o.Lz(),
            'Op': o.Op(),
            'Or': o.Or(),
            'Oz': o.Oz(),
            'R': o.R(),
            'Tp': o.Tp(),
            'Tr': o.Tr(),
            'TrTp': o.TrTp(),
            'Tz': o.Tz(),
            'U': o.U(),
            'V': o.V(),
            'W': o.W(),
            'jp': o.jp(),
            'jr': o.jr(),
            'jz': o.jz()
        })

    except Exception as e:
        # Return NaNs if something fails
        return pd.Series({key: np.nan for key in [
            'rapo', 'rperi', 'eccentricity', 'zmax',
            'E', 'ER', 'Ez', 'LcE', 'Lz',
            'Op', 'Or', 'Oz', 'R',
            'Tp', 'Tr', 'TrTp', 'Tz',
            'U', 'V', 'W',
            'jp', 'jr', 'jz'
        ]})



## Reading samples

### HARPS

In [114]:
row = harps_combined.iloc[0]



o = Orbit([row['ra'], row['dec'], 1/row['parallax'], row['pmra'], row['pmdec'], row['radial_velocity']],
                  radec=True, ro=8.125, zo=0.02, vo=220.)

ts = np.linspace(0, 10, 1000) * u.Gyr

# Integrate orbit
o.integrate(ts, MWPotential2014)

print (o.rap(), o.rperi(), o.e(), o.zmax())



9.48683289136146 5.790126492501633 0.24197919926164255 0.384327791986083


In [115]:
o.rap(), o.rperi(), o.e(), o.zmax(), o.E(), o.ER(), o.Ez(), o.LcE(), o.Lz(), o.Op(), o.Or(), o.Oz(), o.R(), o.Tp(), o.Tr(), o.TrTp(), o.Tz(), o.U(), o.V(), o.W(), o.jp(), o.jr(), o.jz()

(9.48683289136146,
 5.790126492501633,
 0.24197919926164255,
 0.384327791986083,
 -44627.17053499921,
 -44870.57152848784,
 243.4009934886259,
 1702.9757230230878,
 1602.6952016000062,
 29.195684065225567,
 39.722431960793074,
 69.08943175072287,
 8.127990687920917,
 0.21520938824870253,
 0.158177256452506,
 2.3090466028457635,
 0.09094278456145859,
 -76.94861526629195,
 -35.217054279541614,
 -29.23584541014547,
 1602.6952016000062,
 67.19044794546524,
 3.7050592297332696)

In [ ]:
harps_c = pd.read_csv('harps_gaia_c.csv')
harps_mg_si = pd.read_csv('harps_gaia_mg_si.csv')
harps_o = pd.read_csv('harps_gaia_o.csv')

print (f'harps_c contains {len(harps_c)} stars')
print (f'harps_mg_si contains {len(harps_mg_si)} stars')    
print (f'harps_o contains {len(harps_o)} stars')

# Ensure 'source_id' is int and set as index
for df in [harps_mg_si, harps_c, harps_o]:
    df['source_id'] = df['source_id'].astype(int)
    df.set_index('source_id', inplace=True)

# Start with the base DataFrame
harps_combined = harps_mg_si.copy()

# Add only new columns from harps_c
new_cols_c = harps_c.columns.difference(harps_combined.columns)
harps_combined = harps_combined.join(harps_c[new_cols_c], how='left')

# Add only new columns from harps_o
new_cols_o = harps_o.columns.difference(harps_combined.columns)
harps_combined = harps_combined.join(harps_o[new_cols_o], how='left')


# Apply to DataFrame
harps_combined[
    ['rapo', 'rperi', 'eccentricity', 'zmax', 'E', 'ER', 'Ez', 'LcE', 'Lz',
     'Op', 'Or', 'Oz', 'R', 'Tp', 'Tr', 'TrTp', 'Tz', 'U', 'V', 'W', 'jp', 'jr', 'jz']
] = harps_combined.progress_apply(
    lambda row: compute_orbit_params(
        row['ra'], row['dec'], row['parallax'],
        row['pmra'], row['pmdec'], row['radial_velocity']
    ), axis=1)



# Bring source_id back as a column
harps_combined.reset_index(inplace=True)
harps_combined.drop(['Unnamed: 0'], axis = 1, inplace = True)
harps_combined.to_csv('harps_sample.csv')

harps_combined.head(2)

harps_c contains 627 stars
harps_mg_si contains 798 stars
harps_o contains 232 stars


100%|██████████| 798/798 [01:47<00:00,  7.44it/s]


,source_id,Star,ra,dec,parallax,parallax_error,pmra,pmra_error,pmdec,pmdec_error,...,Tp,Tr,TrTp,Tz,U,V,W,jp,jr,jz
0,3790755044736796544,BD-012505,169.304406,-1.981918,30.138378,0.023513,-568.299830,0.017789,-14.309893,0.014746,...,0.215211,0.158178,2.309044,0.090947,-76.948615,-35.217054,-29.235845,1602.694743,67.192390,3.707134
1,4706630501049679744,HD55,1.322329,-67.835081,61.006400,0.019343,-116.617908,0.019904,-560.587881,0.018929,...,0.196349,0.143201,2.291224,0.071753,35.732556,-48.103695,-0.865637,1494.455057,59.348105,0.279256


### APOGEE

In [128]:
apogee = pd.read_csv('apogeeDR17_MST_final_sample.csv')

# Mapping from apogee column names to harps column names
apogee_rename_map = {
    'RA': 'ra',
    'DEC': 'dec',
    'GAIAEDR3_SOURCE_ID': 'source_id',
    'GAIAEDR3_PARALLAX': 'parallax',
    'GAIAEDR3_PARALLAX_ERROR': 'parallax_error',
    'GAIAEDR3_PMRA': 'pmra',
    'GAIAEDR3_PMRA_ERROR': 'pmra_error',
    'GAIAEDR3_PMDEC': 'pmdec',
    'GAIAEDR3_PMDEC_ERROR': 'pmdec_error',
    'GAIAEDR3_PHOT_G_MEAN_MAG': 'phot_g_mean_mag',
    'GAIAEDR3_PHOT_BP_MEAN_MAG': 'phot_bp_mean_mag',
    'GAIAEDR3_PHOT_RP_MEAN_MAG': 'phot_rp_mean_mag',
    'GAIAEDR3_R_MED_PHOTOGEO': 'distance_gspphot',  
    'TEFF': 'Teff',
    'TEFF_ERR': 'e_Teff',
    'LOGG': 'logg',
    'LOGG_ERR': 'e_logg',
    'CI_H': 'c_h', # this is atomic carbon abundance
    'CI_H_ERR': 'e_c_h',
    'O_H': 'o_h',
    'O_H_ERR': 'e_o_h',
    'MG_H': 'mg_h',
    'MG_H_ERR': 'e_mg_h',
    'SI_H': 'si_h',
    'SI_H_ERR': 'e_si_h',
    'FE_H': 'fe_h',
    'FE_H_ERR': 'e_fe_h',
    'VHELIO_AVG': 'radial_velocity',
    'VERR': 'radial_velocity_error',
    'U_lsr': 'U_lsr',
    'V_lsr': 'V_lsr',
    'W_lsr': 'W_lsr',
    'P_gal': 'P_gal'
}

# Rename apogee columns
apogee_renamed = apogee.rename(columns=apogee_rename_map)
apogee_renamed.drop(['Unnamed: 0'], axis = 1, inplace = True)


apogee_renamed[
    ['rapo', 'rperi', 'eccentricity', 'zmax', 'E', 'ER', 'Ez', 'LcE', 'Lz',
     'Op', 'Or', 'Oz', 'R', 'Tp', 'Tr', 'TrTp', 'Tz', 'U', 'V', 'W', 'jp', 'jr', 'jz']
] = apogee_renamed.progress_apply(
    lambda row: compute_orbit_params(
        row['ra'], row['dec'], row['parallax'],
        row['pmra'], row['pmdec'], row['radial_velocity']
    ), axis=1)


apogee_renamed.to_csv('apogee_sample.csv', index=False)
apogee_renamed.head(2)

  0%|          | 387/108526 [00:37<2:56:06, 10.23it/s]

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:101: RuntimeWarning: overflow encountered in scalar power
  return -self._mass(r) * R / r**3.0

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/MiyamotoNagaiPotential.py:73: RuntimeWarning: overflow encountered in scalar power
  return -R / (R**2.0 + (self._a + numpy.sqrt(z**2.0 + self._b2)) ** 2.0) ** (

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:218: RuntimeWarning: overflow encountered in square
  -((R[~numpy.isinf(R)] / self.rc) ** 2.0),

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:84: RuntimeWarning: overflow encountered in scalar power
  * special.gammainc(1.0 - self.alpha / 2.0, (r / self.rc) ** 2.0)

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:86: RuntimeWarning: ove

  4%|▎         | 3920/108526 [07:18<2:42:56, 10.70it/s]

/tmp/ipykernel_156319/957184328.py:9: RuntimeWarning: divide by zero encountered in scalar divide
  distance = 1.0 / plx  # distance in kpc

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/util/coords.py:512: RuntimeWarning: invalid value encountered in multiply
  return (R.T * invr.T).sum(-1)

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/numpy/core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)



100%|██████████| 108526/108526 [3:23:02<00:00,  8.91it/s] 


,ra,dec,GLON,GLAT,radial_velocity,radial_velocity_error,source_id,parallax,parallax_error,pmra,...,Tp,Tr,TrTp,Tz,U,V,W,jp,jr,jz
0,0.005558,57.354549,116.014776,-4.824917,-101.71,0.07,421086363305436800,0.13,0.03,-1.15,...,0.356782,0.266628,2.347749,0.249784,78.260793,-72.596498,29.678096,2450.148280,22.217181,35.734650
1,0.008802,63.463078,117.227985,1.162167,-106.63,0.03,431594980053422720,0.07,0.04,-0.92,...,0.679407,0.508199,2.349920,0.577328,113.482693,-60.404416,-50.223743,4099.529006,38.309357,68.889448


### Galah

In [ ]:
galah = pd.read_csv('galahDR4_final_sample.csv')
galah.drop(columns = ['Unnamed: 0', 'flag_c_fe'], axis = 1, inplace = True)
galah_c = pd.read_csv('galahDR4_final_carbon_sample.csv')

# Ensure both have the same type for the identifier
galah['gaiadr3_source_id'] = galah['gaiadr3_source_id'].astype(int)
galah_c['gaiadr3_source_id'] = galah_c['gaiadr3_source_id'].astype(int)

# Create a set of IDs from the cleaned DF
cleaned_ids = set(galah_c['gaiadr3_source_id'])

# Add the flag column to galah
galah['c_flag'] = galah['gaiadr3_source_id'].isin(cleaned_ids)

# Mapping from galah column names to harps-style column names
galah_rename_map = {
    'gaiadr3_source_id': 'source_id',
    'rv_comp_1': 'radial_velocity',
    'e_rv_comp_1': 'radial_velocity_error',
    'teff': 'Teff',
    'e_teff': 'e_Teff',
    'logg': 'logg',
    'e_logg': 'e_logg',
    'fe_h': 'fe_h',
    'e_fe_h': 'e_fe_h',
    'c_h': 'c_h',
    'e_c_h': 'e_c_h',
    'o_h': 'o_h',
    'e_o_h': 'e_o_h',
    'mg_h': 'mg_h',
    'e_mg_h': 'e_mg_h',
    'si_h': 'si_h',
    'e_si_h': 'e_si_h',
    'ra': 'ra',
    'dec': 'dec',
    'parallax': 'parallax',
    'parallax_error': 'parallax_error',
    'pmra': 'pmra',
    'pmra_error': 'pmra_error',
    'pmdec': 'pmdec',
    'pmdec_error': 'pmdec_error',
    'phot_g_mean_mag': 'phot_g_mean_mag',
    'phot_bp_mean_mag': 'phot_bp_mean_mag',
    'phot_rp_mean_mag': 'phot_rp_mean_mag',
    'r_med_photogeo': 'distance_gspphot',  # rename to harps-style
    'U_lsr': 'U_lsr',
    'V_lsr': 'V_lsr',
    'W_lsr': 'W_lsr',
    'P_gal': 'P_gal'
}

# Apply the renaming
galah_renamed = galah.rename(columns=galah_rename_map)

galah_renamed[
    ['rapo', 'rperi', 'eccentricity', 'zmax', 'E', 'ER', 'Ez', 'LcE', 'Lz',
     'Op', 'Or', 'Oz', 'R', 'Tp', 'Tr', 'TrTp', 'Tz', 'U', 'V', 'W', 'jp', 'jr', 'jz']
] = galah_renamed.progress_apply(
    lambda row: compute_orbit_params(
        row['ra'], row['dec'], row['parallax'],
        row['pmra'], row['pmdec'], row['radial_velocity']
    ), axis=1)

#galah_renamed.drop(columns = ['Unnamed: 0', 'flag_c_fe'], axis = 1, inplace = True)
galah_renamed.to_csv('galah_sample.csv', index = False)

 42%|████▏     | 48320/116050 [1:43:28<1:49:36, 10.30it/s] 

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:101: RuntimeWarning: overflow encountered in scalar power
  return -self._mass(r) * R / r**3.0

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/MiyamotoNagaiPotential.py:73: RuntimeWarning: overflow encountered in scalar power
  return -R / (R**2.0 + (self._a + numpy.sqrt(z**2.0 + self._b2)) ** 2.0) ** (

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:218: RuntimeWarning: overflow encountered in square
  -((R[~numpy.isinf(R)] / self.rc) ** 2.0),

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:84: RuntimeWarning: overflow encountered in scalar power
  * special.gammainc(1.0 - self.alpha / 2.0, (r / self.rc) ** 2.0)

/home/vadibekyan/anaconda3/lib/python3.12/site-packages/galpy/potential/PowerSphericalPotentialwCutoff.py:86: RuntimeWarning: ove

100%|██████████| 116050/116050 [3:31:24<00:00,  9.15it/s] 


KeyError: "['Unnamed: 0', 'flag_c_fe'] not found in axis"

In [131]:
gr8 = pd.read_csv('gr8_not_in_harps_sample_tmp.csv')


gr8[
    ['rapo', 'rperi', 'eccentricity', 'zmax', 'E', 'ER', 'Ez', 'LcE', 'Lz',
     'Op', 'Or', 'Oz', 'R', 'Tp', 'Tr', 'TrTp', 'Tz', 'U', 'V', 'W', 'jp', 'jr', 'jz']
] = gr8.progress_apply(
    lambda row: compute_orbit_params(
        row['ra'], row['dec'], row['parallax'],
        row['pmra'], row['pmdec'], row['radial_velocity']
    ), axis=1)

gr8.to_csv('gr8_not_in_harps_sample.csv', index = False)



100%|██████████| 1430/1430 [03:02<00:00,  7.85it/s]
